# Технологии обработки больших данных

## Лабораторная работа 6. DevOps для Big Data

**Вариант:** 1. Сервис рекомендаций по отзывам Amazon, категория All_Beauty.

**Уникальное требование:** 80% мощностей — spot-инстансы, и при их отзыве расчёт не теряет уже принятые события. Ориентир доступности сценария рекомендаций из методички — 99.5%.

**Цель:** посчитать стоимость до и после перевода на spot, описать отказоустойчивость и приложить манифесты контейнера и Kubernetes. Живой кластер из Colab не поднимается: числа считаются здесь, а YAML проверяется как текст.


## Нагрузка, от которой считаем ресурсы


In [1]:
import os
from pathlib import Path

try:
    import google.colab  # noqa: F401
    BASE = Path("/content/lab6_devops")
except ImportError:
    BASE = Path(os.environ.get("TEMP", "/tmp")) / "lab6_devops"
BASE.mkdir(parents=True, exist_ok=True)

# Параметры расчёта. Это не измерения из файла отзывов, а тариф и схема сервиса.
REPLICAS = 10
SPOT_SHARE = 0.80
ONDEMAND_RUB_PER_HOUR = 7.50
SPOT_DISCOUNT = 0.70
HOURS_MONTH = 24 * 30
SPOT_INTERRUPT_PER_HOUR = 0.05
FAILOVER_SEC = 20
SLA = 0.995

spot_replicas = int(REPLICAS * SPOT_SHARE)
ondemand_replicas = REPLICAS - spot_replicas
print(f"Реплик всего {REPLICAS}: spot {spot_replicas}, on-demand {ondemand_replicas}")


Реплик всего 10: spot 8, on-demand 2


## Стоимость и бюджет ошибок

Spot дешевле, но его могут забрать. Чекпоинт из потоковой обработки ограничивает потерю одним коротким окном, а две обычные реплики не дают сервису остаться вообще без машин.


In [2]:
cost_before = REPLICAS * ONDEMAND_RUB_PER_HOUR * HOURS_MONTH
spot_price = ONDEMAND_RUB_PER_HOUR * (1 - SPOT_DISCOUNT)
cost_after = (ondemand_replicas * ONDEMAND_RUB_PER_HOUR + spot_replicas * spot_price) * HOURS_MONTH
savings = (cost_before - cost_after) / cost_before

interrupts = spot_replicas * SPOT_INTERRUPT_PER_HOUR * HOURS_MONTH
downtime_min = interrupts * FAILOVER_SEC / 60
budget_min = (1 - SLA) * HOURS_MONTH * 60
sla_ok = downtime_min < budget_min

print(f"Было, только on-demand: {cost_before:,.0f} ₽/мес")
print(f"Стало, {SPOT_SHARE:.0%} spot: {cost_after:,.0f} ₽/мес")
print(f"Экономия: {savings:.1%}")
print(f"Ожидаемых отзывов spot за месяц: {interrupts:.0f}")
print(f"Простой на перезапуск: {downtime_min:.1f} мин при бюджете {budget_min:.1f} мин")
print("Укладываемся в 99.5%:", sla_ok)
print(
    "Вывод по стоимости: восемь из десяти реплик на spot дают экономию больше половины, "
    "пока отзыв машины означает перезапуск пода с чекпоинта, а не пересчёт архива. "
    "Две on-demand реплики держат выдачу, когда spot забирают пачкой."
)


Было, только on-demand: 54,000 ₽/мес
Стало, 80% spot: 23,760 ₽/мес
Экономия: 56.0%
Ожидаемых отзывов spot за месяц: 288
Простой на перезапуск: 96.0 мин при бюджете 216.0 мин
Укладываемся в 99.5%: True
Вывод по стоимости: восемь из десяти реплик на spot дают экономию больше половины, пока отзыв машины означает перезапуск пода с чекпоинта, а не пересчёт архива. Две on-demand реплики держат выдачу, когда spot забирают пачкой.


## Контейнер и манифесты

Образ — обычный Python-процесс выдачи. В Kubernetes два Deployment: постоянный и spot с допуском к taint. HPA крутит только spot-группу. В CI — проверка, что в манифесте действительно 8 spot и 2 обычных реплики.


In [3]:
dockerfile = """
FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY app.py .
CMD ["python", "app.py"]
""".strip() + "\n"

app_py = """
import json
import os
from http.server import BaseHTTPRequestHandler, HTTPServer

CHECKPOINT = os.environ.get("CHECKPOINT_PATH", "/data/checkpoint.json")

def load_state():
    if not os.path.exists(CHECKPOINT):
        return {"served": 0}
    with open(CHECKPOINT, encoding="utf-8") as src:
        return json.load(src)

class Handler(BaseHTTPRequestHandler):
    def do_GET(self):
        state = load_state()
        body = json.dumps({"status": "ok", "served": state.get("served", 0)}).encode()
        self.send_response(200)
        self.send_header("Content-Type", "application/json")
        self.end_headers()
        self.wfile.write(body)

    def log_message(self, fmt, *args):
        return

if __name__ == "__main__":
    HTTPServer(("0.0.0.0", 8080), Handler).serve_forever()
""".strip() + "\n"

requirements = "\n".join([]) + "\n"

deployment = f"""
apiVersion: apps/v1
kind: Deployment
metadata:
  name: rec-ondemand
spec:
  replicas: {ondemand_replicas}
  selector:
    matchLabels:
      app: rec
      pool: ondemand
  template:
    metadata:
      labels:
        app: rec
        pool: ondemand
    spec:
      containers:
        - name: rec
          image: rec-service:lab6
          ports:
            - containerPort: 8080
          readinessProbe:
            httpGet:
              path: /
              port: 8080
          resources:
            requests:
              cpu: "500m"
              memory: 512Mi
          volumeMounts:
            - name: checkpoint
              mountPath: /data
      volumes:
        - name: checkpoint
          emptyDir: {{}}
---
apiVersion: apps/v1
kind: Deployment
metadata:
  name: rec-spot
spec:
  replicas: {spot_replicas}
  selector:
    matchLabels:
      app: rec
      pool: spot
  template:
    metadata:
      labels:
        app: rec
        pool: spot
    spec:
      tolerations:
        - key: "cloud.google.com/gke-spot"
          operator: "Equal"
          value: "true"
          effect: "NoSchedule"
      affinity:
        nodeAffinity:
          requiredDuringSchedulingIgnoredDuringExecution:
            nodeSelectorTerms:
              - matchExpressions:
                  - key: node-pool
                    operator: In
                    values: ["spot"]
      containers:
        - name: rec
          image: rec-service:lab6
          ports:
            - containerPort: 8080
          resources:
            requests:
              cpu: "500m"
              memory: 512Mi
          volumeMounts:
            - name: checkpoint
              mountPath: /data
      volumes:
        - name: checkpoint
          emptyDir: {{}}
""".strip() + "\n"

hpa = """
apiVersion: autoscaling/v2
kind: HorizontalPodAutoscaler
metadata:
  name: rec-spot
spec:
  scaleTargetRef:
    apiVersion: apps/v1
    kind: Deployment
    name: rec-spot
  minReplicas: 4
  maxReplicas: 12
  metrics:
    - type: Resource
      resource:
        name: cpu
        target:
          type: Utilization
          averageUtilization: 70
""".strip() + "\n"

pdb = f"""
apiVersion: policy/v1
kind: PodDisruptionBudget
metadata:
  name: rec
spec:
  minAvailable: {ondemand_replicas}
  selector:
    matchLabels:
      app: rec
""".strip() + "\n"

ci = """
name: lab6
on: [push]
jobs:
  check:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - name: Spot share
        run: python check_manifests.py
""".strip() + "\n"

check_py = f"""
from pathlib import Path
text = Path("deployment.yaml").read_text(encoding="utf-8")
assert "replicas: {spot_replicas}" in text
assert "replicas: {ondemand_replicas}" in text
assert "checkpoint" in text
print("manifest ok")
""".strip() + "\n"

files = {
    "Dockerfile": dockerfile,
    "app.py": app_py,
    "requirements.txt": requirements,
    "deployment.yaml": deployment,
    "hpa.yaml": hpa,
    "pdb.yaml": pdb,
    "ci.yml": ci,
    "check_manifests.py": check_py,
}
for name, body in files.items():
    (BASE / name).write_text(body, encoding="utf-8")

# Проверка текста манифеста теми же условиями, что и CI
ns = {}
exec(check_py.replace('Path("deployment.yaml")', 'Path(r"%s")' % (BASE / "deployment.yaml")), ns)
print("Файлы:", ", ".join(sorted(files)))
print("Проверка доли spot прошла")


manifest ok
Файлы: Dockerfile, app.py, check_manifests.py, ci.yml, deployment.yaml, hpa.yaml, pdb.yaml, requirements.txt
Проверка доли spot прошла


## Результаты


In [4]:
print("Вывод")
print(
    f"Для сервиса рекомендаций взял {REPLICAS} реплик и перевёл {spot_replicas} из них на spot, "
    f"это ровно {SPOT_SHARE:.0%}. При цене on-demand {ONDEMAND_RUB_PER_HOUR:.2f} ₽/час и скидке spot {SPOT_DISCOUNT:.0%} "
    f"счёт падает с {cost_before:,.0f} до {cost_after:,.0f} ₽ в месяц, экономия {savings:.1%}. "
    f"Ожидаемый простой на перезапуск {downtime_min:.0f} мин меньше бюджета {budget_min:.0f} мин для доступности {SLA:.1%}. "
    "Отказоустойчивость держится на чекпоинте, двух обычных репликах и PodDisruptionBudget. "
    "Манифесты Deployment, HPA и PDB записаны рядом с Dockerfile; отдельный кластер в этой среде не поднимался."
)


Вывод
Для сервиса рекомендаций взял 10 реплик и перевёл 8 из них на spot, это ровно 80%. При цене on-demand 7.50 ₽/час и скидке spot 70% счёт падает с 54,000 до 23,760 ₽ в месяц, экономия 56.0%. Ожидаемый простой на перезапуск 96 мин меньше бюджета 216 мин для доступности 99.5%. Отказоустойчивость держится на чекпоинте, двух обычных репликах и PodDisruptionBudget. Манифесты Deployment, HPA и PDB записаны рядом с Dockerfile; отдельный кластер в этой среде не поднимался.
